# AstroHealth: train the ECG beat classifier (MIT-BIH, Colab)

Trains a small 1D-CNN to classify single heartbeats (5 AAMI classes) on the **MIT-BIH Arrhythmia Database**, then exports it for TensorFlow.js.

What this fixes compared with a minimal tutorial notebook: it actually **trains** on labelled beats, splits train/test **by patient**, and exports with a loader that TF.js accepts (the Keras 3 `to_json()` format does not load).

**Run:** Runtime > Run all. The download is about 100 MB. At the end, `ecg_model.zip` downloads.

> Demo-grade decision support only, not a diagnostic device. MIT-BIH is a research dataset of 47 subjects. Cite PhysioNet and Moody & Mark (2001) when you use it.

In [ ]:
import os, json, shutil
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
import wfdb
from sklearn.metrics import classification_report, confusion_matrix

SEED = 42
np.random.seed(SEED); tf.random.set_seed(SEED)
DATA_DIR = "mitdb"
print("TensorFlow", tf.__version__)

## 1. Download MIT-BIH

In [ ]:
if not os.path.exists(f"{DATA_DIR}/100.hea"):
    os.makedirs(DATA_DIR, exist_ok=True)
    print("Downloading mitdb (about 100 MB) ...")
    wfdb.dl_database("mitdb", dl_dir=DATA_DIR)
print("records on disk:", len([f for f in os.listdir(DATA_DIR) if f.endswith(".hea")]))

## 2. Settings: classes and patient-wise split
Beat labels are grouped into the 5 standard AAMI classes. Records 102, 104, 107, 217 (paced rhythms) are left out, as is standard. Train and test use **different patients** (the common "inter-patient" DS1/DS2 split), which is a much more honest test than a random split of beats.

In [ ]:
CLASS_NAMES = ["N", "S", "V", "F", "Q"]
CLASS_DESC = {"N": "Normal / bundle-branch beat", "S": "Supraventricular ectopic",
              "V": "Ventricular ectopic", "F": "Fusion", "Q": "Unknown / paced"}
AAMI = {}
for sym in ["N", "L", "R", "e", "j"]: AAMI[sym] = 0
for sym in ["A", "a", "J", "S"]:      AAMI[sym] = 1
for sym in ["V", "E"]:                AAMI[sym] = 2
AAMI["F"] = 3
for sym in ["/", "f", "Q"]:           AAMI[sym] = 4

DS1 = ["101","106","108","109","112","114","115","116","118","119","122","124",
       "201","203","205","207","208","209","215","220","223","230"]
DS2 = ["100","103","105","111","113","117","121","123","200","202","210","212",
       "213","214","219","221","222","228","231","232","233","234"]
VAL_RECORDS = ["112", "122", "209", "223"]          # held out of DS1 for early stopping
TRAIN_RECORDS = [r for r in DS1 if r not in VAL_RECORDS]

PRE, POST = 90, 90                                   # 90 + 90 = 180 samples at 360 Hz

## 3. Cut every annotated beat into a 180-sample window
Each window is centred on the R-peak. Every window is standardised on its own (subtract its mean, divide by its standard deviation), so the app needs no stored statistics. It only has to do the same per-window step.

In [ ]:
def load_beats(rec_name):
    path = f"{DATA_DIR}/{rec_name}"
    rec = wfdb.rdrecord(path)
    ann = wfdb.rdann(path, "atr")
    if "MLII" not in rec.sig_name:
        return np.zeros((0, PRE + POST, 1), "float32"), np.zeros((0,), "int32")
    sig = rec.p_signal[:, rec.sig_name.index("MLII")]      # some records list channels in a different order
    X, y = [], []
    for s, sym in zip(ann.sample, ann.symbol):
        cls = AAMI.get(sym)
        if cls is None or s - PRE < 0 or s + POST > len(sig):
            continue
        w = sig[s - PRE: s + POST]
        sd = w.std()
        if sd < 1e-6:
            continue
        X.append((w - w.mean()) / sd); y.append(cls)
    return np.array(X, "float32").reshape(-1, PRE + POST, 1), np.array(y, "int32")

def build(records):
    Xs, ys = [], []
    for r in records:
        X, y = load_beats(r)
        Xs.append(X); ys.append(y)
    return np.concatenate(Xs), np.concatenate(ys)

X_train, y_train = build(TRAIN_RECORDS)
X_val,   y_val   = build(VAL_RECORDS)
X_test,  y_test  = build(DS2)
print("train / val / test beats:", len(y_train), len(y_val), len(y_test))
for name, y in [("train", y_train), ("test", y_test)]:
    print(name, dict(zip(CLASS_NAMES, np.bincount(y, minlength=5))))

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(14, 2.4), sharey=True)
for c, ax in enumerate(axes):
    idx = np.where(y_train == c)[0][:8]
    for i in idx: ax.plot(X_train[i, :, 0], alpha=0.5)
    ax.set_title(f"{CLASS_NAMES[c]}"); ax.set_xticks([])
plt.tight_layout(); plt.show()

## 4. Model and training
Same small CNN idea as the basic tutorial (two Conv1D blocks, then Dense), with dropout. Normal beats outnumber everything else, so class weights (softened with a square root) keep the rare classes from being ignored.

In [ ]:
def build_model():
    m = keras.Sequential([
        keras.layers.Input(shape=(PRE + POST, 1)),
        keras.layers.Conv1D(32, 5, activation="relu"),
        keras.layers.MaxPooling1D(2),
        keras.layers.Conv1D(64, 5, activation="relu"),
        keras.layers.MaxPooling1D(2),
        keras.layers.Flatten(),
        keras.layers.Dense(64, activation="relu"),
        keras.layers.Dropout(0.3),
        keras.layers.Dense(5, activation="softmax"),
    ])
    m.compile(optimizer=keras.optimizers.Adam(1e-3),
              loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return m

counts = np.bincount(y_train, minlength=5).astype("float64")
balanced = counts.sum() / (5 * np.maximum(counts, 1))
class_weight = dict(enumerate(np.sqrt(balanced)))
print("class weights:", {CLASS_NAMES[k]: round(v, 2) for k, v in class_weight.items()})

model = build_model()
model.summary()
history = model.fit(
    X_train, y_train, validation_data=(X_val, y_val),
    epochs=30, batch_size=256, class_weight=class_weight, verbose=2,
    callbacks=[keras.callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)])

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 3.2))
ax[0].plot(history.history["loss"], label="train"); ax[0].plot(history.history["val_loss"], label="val"); ax[0].set_title("Loss"); ax[0].legend()
ax[1].plot(history.history["accuracy"], label="train"); ax[1].plot(history.history["val_accuracy"], label="val"); ax[1].set_title("Accuracy"); ax[1].legend()
plt.tight_layout(); plt.show()

## 5. Test on patients the model has never seen
Do not rely on overall accuracy. Normal beats are most of the data, so a model can score well while missing the rare, clinically important classes. Look at **recall per class**. The supraventricular class (S) is known to be the hardest under this inter-patient split, so expect it to be weak.

In [ ]:
pred = model.predict(X_test, batch_size=1024, verbose=0).argmax(1)
acc = float((pred == y_test).mean())
print(f"TEST accuracy (unseen patients): {acc:.4f}\n")
print(classification_report(y_test, pred, labels=range(5), target_names=CLASS_NAMES, digits=3, zero_division=0))

cm = confusion_matrix(y_test, pred, labels=range(5))
fig, ax = plt.subplots(figsize=(5, 4.5))
ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(5)); ax.set_xticklabels(CLASS_NAMES); ax.set_yticks(range(5)); ax.set_yticklabels(CLASS_NAMES)
for i in range(5):
    for j in range(5):
        ax.text(j, i, cm[i, j], ha="center", va="center", color="white" if cm[i, j] > cm.max() / 2 else "black", fontsize=8)
ax.set_xlabel("Predicted"); ax.set_ylabel("True"); ax.set_title("Confusion matrix (test)")
plt.tight_layout(); plt.show()

## 6. Export for TensorFlow.js
This writes the TF.js format directly. It does not use the `tensorflowjs` converter (fails on Keras 3), and it does not use `model.to_json()` (Keras 3 format that TF.js cannot read).

In [ ]:
_KI = {"class_name": "VarianceScaling", "config": {"scale": 1, "mode": "fan_avg", "distribution": "normal", "seed": None}}
_BI = {"class_name": "Zeros", "config": {}}
_NONE = dict(kernel_regularizer=None, bias_regularizer=None, activity_regularizer=None,
             kernel_constraint=None, bias_constraint=None)

def export_tfjs(model, out_dir, meta=None):
    os.makedirs(out_dir, exist_ok=True)
    in_shape = [None] + [int(d) for d in model.input_shape[1:]]
    layers, specs, chunks, counts, first = [], [], [], {}, True

    def nm(kind):
        counts[kind] = counts.get(kind, 0) + 1
        return f"{kind.lower()}_{kind}{counts[kind]}"

    def add_weights(name, layer):
        kernel, bias = [w.astype("<f4") for w in layer.get_weights()]
        specs.extend([{"name": f"{name}/kernel", "shape": list(kernel.shape), "dtype": "float32"},
                      {"name": f"{name}/bias", "shape": list(bias.shape), "dtype": "float32"}])
        chunks.extend([kernel.tobytes(), bias.tobytes()])

    for layer in model.layers:
        if isinstance(layer, keras.layers.InputLayer):
            continue
        cls = type(layer).__name__
        extra = {}
        if first:
            extra = {"batch_input_shape": in_shape, "dtype": "float32"}; first = False
        if cls == "Conv1D":
            name = nm("Conv1D")
            cfg = {"filters": int(layer.filters), "kernel_size": [int(layer.kernel_size[0])],
                   "strides": [int(layer.strides[0])], "padding": layer.padding,
                   "data_format": "channels_last", "dilation_rate": [int(layer.dilation_rate[0])],
                   "activation": layer.activation.__name__, "use_bias": True,
                   "kernel_initializer": _KI, "bias_initializer": _BI, **_NONE,
                   "name": name, "trainable": True, **extra}
            add_weights(name, layer)
        elif cls == "Dense":
            name = nm("Dense")
            cfg = {"units": int(layer.units), "activation": layer.activation.__name__, "use_bias": True,
                   "kernel_initializer": _KI, "bias_initializer": _BI, **_NONE,
                   "name": name, "trainable": True, **extra}
            add_weights(name, layer)
        elif cls == "MaxPooling1D":
            cfg = {"pool_size": [int(layer.pool_size[0])], "padding": layer.padding,
                   "strides": [int(layer.strides[0])], "name": nm("MaxPooling1D"), "trainable": True, **extra}
        elif cls == "GlobalAveragePooling1D":
            cfg = {"name": nm("GlobalAveragePooling1D"), "trainable": True, **extra}
        elif cls == "Flatten":
            cfg = {"name": nm("Flatten"), "trainable": True, **extra}
        elif cls == "Dropout":
            cfg = {"rate": float(layer.rate), "noise_shape": None, "seed": None,
                   "name": nm("Dropout"), "trainable": True, **extra}
        else:
            raise ValueError(f"Layer type {cls} is not supported by this exporter")
        layers.append({"class_name": cls, "config": cfg})

    json.dump({"modelTopology": {"class_name": "Sequential",
                                 "config": {"name": "exported_model", "layers": layers},
                                 "keras_version": "2.15.0", "backend": "tensorflow"},
               "format": "layers-model", "generatedBy": "manual exporter", "convertedBy": None,
               "weightsManifest": [{"paths": ["weights.bin"], "weights": specs}]},
              open(f"{out_dir}/model.json", "w"))
    open(f"{out_dir}/weights.bin", "wb").write(b"".join(chunks))
    if meta is not None:
        json.dump(meta, open(f"{out_dir}/meta.json", "w"), indent=2)

OUT = "ecg_model"
export_tfjs(model, OUT, meta={
    "classNames": CLASS_NAMES, "classDescriptions": CLASS_DESC,
    "sampleRate": 360, "windowSamples": PRE + POST, "preSamples": PRE, "postSamples": POST,
    "normalization": "per-window z-score: (x - mean(x)) / std(x)",
    "lead": "MLII", "testAccuracyUnseenPatients": acc, "syntheticData": False,
})
n_bytes = os.path.getsize(f"{OUT}/weights.bin")
assert n_bytes == model.count_params() * 4, "weights.bin size does not match parameter count"
print("Exported:", sorted(os.listdir(OUT)), f"| weights {n_bytes / 1024:.0f} KB (size check OK)")

In [ ]:
shutil.make_archive("ecg_model", "zip", ".", OUT)
print("Created:", os.path.abspath("ecg_model.zip"), f"({os.path.getsize('ecg_model.zip') / 1024:.0f} KB)")
try:
    from google.colab import files
    files.download("ecg_model.zip")
except Exception as e:
    print("Automatic download did not start:", e)
    print("Use the folder icon on the left > refresh > three dots next to ecg_model.zip > Download.")

## 7. Use it in the app
Place the three files in `public/models/ecg-cnn/` (`model.json`, `weights.bin`, `meta.json`), then:

```js
import * as tf from '@tensorflow/tfjs';
const base = import.meta.env.BASE_URL;
const model = await tf.loadLayersModel(`${base}models/ecg-cnn/model.json`);
const meta  = await (await fetch(`${base}models/ecg-cnn/meta.json`)).json();

// beat: array of 180 samples at 360 Hz, centred on the R-peak
function classifyBeat(beat) {
  const mean = beat.reduce((a, b) => a + b, 0) / beat.length;
  const sd = Math.sqrt(beat.reduce((a, b) => a + (b - mean) ** 2, 0) / beat.length) || 1;
  const x = tf.tensor3d([beat.map(v => [(v - mean) / sd])]);       // [1, 180, 1]
  const p = model.predict(x).dataSync();
  x.dispose();
  return meta.classNames.map((name, i) => ({ name, probability: p[i] }));
}
```

The app must (1) find R-peaks in the live signal, (2) resample to 360 Hz if needed, (3) cut 90 samples before and 90 after each peak, and (4) apply the same per-window standardisation. If any of these differ from training, the results will be wrong.